# 02 — PhoBERT (`vinai/phobert-base-v2`)

Notebook PhoBERT được viết lại hoàn toàn cho project phát hiện ngôn ngữ độc hại tiếng Việt.

## Tác vụ
- **Task 1:** `NON-TOXIC` vs `TOXIC`
- **Task 2:** `EXPLICIT` vs `IMPLICIT`, chỉ trên các mẫu có **gold `toxicity == TOXIC`**

## Protocol khóa
- max epochs = **10**
- early stopping patience = **3**
- best checkpoint = **Dev Macro-F1**
- AdamW, LR `2e-5`, weight decay `0.01`
- warmup `10%`, cosine scheduler
- gradient clipping `1.0`
- weighted Cross Entropy, weights chỉ từ Train
- mixed precision FP16 trên T4
- effective batch mục tiêu = `32`
- seed = `42`
- Test chỉ chạy **sau khi restore best Dev checkpoint**
- `evidence` tuyệt đối không được dùng làm model input

## Điều kiện trước khi chạy
1. Đã chạy `00_Prepare_Data.ipynb`.
2. Drive có `MyDrive/DS108_Toxic_4Models/data/` và `data_splits/`.
3. Colab đã bật **T4 GPU**.

Notebook này **không dùng `py_vncorenlp.download_model()`** để tránh lỗi `VnCoreNLP-1.2.jar not found`. Nó tải trực tiếp các file word-segmentation cần thiết từ repository VnCoreNLP chính thức.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. Chỉ cài các package thực sự cần cho PhoBERT.
# Warning dependency liên quan Gradio của môi trường Colab có thể bỏ qua
# vì notebook này không dùng Gradio.
!pip -q install \
  "transformers==4.49.0" \
  "scikit-learn==1.6.1" \
  "pandas==2.2.2" \
  "tqdm==4.67.1" \
  "py_vncorenlp==0.1.4"

!java -version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
[0.019s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyte

In [ ]:
# 3. Imports + cấu hình
import os, gc, sys, json, math, time, random, shutil, hashlib, platform
from pathlib import Path
import urllib.request
import traceback

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import sklearn
import transformers
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    average_precision_score,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from tqdm.auto import tqdm
import py_vncorenlp

SEED = 42
MODEL_KEY = 'phobert'
MODEL_ID = 'vinai/phobert-base-v2'
MAX_EPOCHS = 10
PATIENCE = 3
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0
MODEL_HARD_MAX = 256

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Hãy bật Runtime → Change runtime type → T4 GPU.'

ROOT = Path('/content/drive/MyDrive/DS108_Toxic_4Models')
DATA_DIR = ROOT / 'data'
SPLIT_DIR = ROOT / 'data_splits'
ARTIFACT_DIR = ROOT / 'artifacts'
RESULT_DIR = ROOT / 'results'
PRED_DIR = RESULT_DIR / 'predictions'
CM_DIR = RESULT_DIR / 'confusion_matrices'
CACHE_DIR = ROOT / 'cache'
for p in [DATA_DIR, SPLIT_DIR, ARTIFACT_DIR, RESULT_DIR, PRED_DIR, CM_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TASKS = {
    'task1': {
        'label_col':'toxicity',
        'label2id':{'NON-TOXIC':0,'TOXIC':1},
        'id2label':{0:'NON-TOXIC',1:'TOXIC'},
    },
    'task2': {
        'label_col':'expression_type',
        'label2id':{'EXPLICIT':0,'IMPLICIT':1},
        'id2label':{0:'EXPLICIT',1:'IMPLICIT'},
    }
}

def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_all()
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))

Python: 3.12.13
PyTorch: 2.11.0+cu128
Transformers: 4.49.0
CUDA: 12.8
GPU: Tesla T4


In [ ]:
# 4. Validate dataset + master split
DATA_PATH = DATA_DIR / 'gold_standard_final_48670.csv'
META_PATH = SPLIT_DIR / 'split_metadata.json'
required = [DATA_PATH, META_PATH] + [SPLIT_DIR/f'{s}.csv' for s in ['train','dev','test']]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f'Thiếu file. Chạy 00_Prepare_Data.ipynb trước. Missing={missing}'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b = f.read(1024*1024)
            if not b: break
            h.update(b)
    return h.hexdigest()

dataset_sha = sha256_file(DATA_PATH)
meta = json.loads(META_PATH.read_text(encoding='utf-8'))
assert meta['dataset_sha256'] == dataset_sha, 'Dataset hiện tại khác dataset đã dùng để tạo split.'

gold = pd.read_csv(DATA_PATH)
assert len(gold) == 48670 and gold['row_id'].is_unique
assert set(gold['toxicity'].unique()) == {'NON-TOXIC','TOXIC'}
tox = gold['toxicity'].eq('TOXIC')
assert set(gold.loc[tox,'expression_type'].unique()) == {'EXPLICIT','IMPLICIT'}
assert gold.loc[~tox,'expression_type'].isna().all()

splits = {s: pd.read_csv(SPLIT_DIR/f'{s}.csv') for s in ['train','dev','test']}
expected = {
    'train':(38936,34248,4688,3999,689),
    'dev':(4867,4281,586,500,86),
    'test':(4867,4281,586,500,86),
}
rows=[]
for s,df in splits.items():
    t=df[df['toxicity'].eq('TOXIC')]
    got=(len(df),int((df['toxicity']=='NON-TOXIC').sum()),int((df['toxicity']=='TOXIC').sum()),int((t['expression_type']=='EXPLICIT').sum()),int((t['expression_type']=='IMPLICIT').sum()))
    assert got == expected[s], (s,got)
    rows.append({'split':s,'total':got[0],'NON-TOXIC':got[1],'TOXIC':got[2],'EXPLICIT':got[3],'IMPLICIT':got[4]})

train_ids=set(splits['train']['row_id']); dev_ids=set(splits['dev']['row_id']); test_ids=set(splits['test']['row_id'])
assert train_ids.isdisjoint(dev_ids) and train_ids.isdisjoint(test_ids) and dev_ids.isdisjoint(test_ids)
assert len(train_ids|dev_ids|test_ids)==48670

display(pd.DataFrame(rows))
print('DATA + MASTER SPLIT: PASS')
print('Dataset SHA256:', dataset_sha)

,split,total,NON-TOXIC,TOXIC,EXPLICIT,IMPLICIT
0,train,38936,34248,4688,3999,689
1,dev,4867,4281,586,500,86
2,test,4867,4281,586,500,86


DATA + MASTER SPLIT: PASS
Dataset SHA256: 0ec2c654eef909051b21b0c30bb970c8bcd374bfac355710cc553f47ea8e9163


## VnCoreNLP word segmentation

PhoBERT yêu cầu input đã được word-segment. Cell dưới đây:
- tải trực tiếp `VnCoreNLP-1.2.jar`, `vi-vocab`, `wordsegmenter.rdr`;
- kiểm tra file;
- test `word_segment()`;
- cache segmentation trên Drive;
- có thể resume nếu Colab ngắt giữa chừng.

In [ ]:
# 5. VnCoreNLP: download trực tiếp từ repository chính thức
VNCORE_DIR = Path('/content/vncorenlp')
WORDSEG_DIR = VNCORE_DIR / 'models' / 'wordsegmenter'
VNCORE_FILES = {
    VNCORE_DIR/'VnCoreNLP-1.2.jar': 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar',
    WORDSEG_DIR/'vi-vocab': 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/vi-vocab',
    WORDSEG_DIR/'wordsegmenter.rdr': 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/wordsegmenter.rdr',
}
MIN_SIZE = {'VnCoreNLP-1.2.jar':10_000_000,'vi-vocab':100_000,'wordsegmenter.rdr':100_000}

def fetch(url,dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size >= MIN_SIZE[dest.name]:
        print('Reuse:', dest)
        return
    if dest.exists(): dest.unlink()
    req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
    print('Downloading:', dest.name)
    with urllib.request.urlopen(req, timeout=180) as r, open(dest,'wb') as f:
        shutil.copyfileobj(r,f)
    assert dest.stat().st_size >= MIN_SIZE[dest.name], f'File tải lỗi: {dest}'

for dest,url in VNCORE_FILES.items():
    fetch(url,dest)

for p in VNCORE_FILES:
    print('PASS', p, round(p.stat().st_size/1024/1024,3),'MB')

segmenter = py_vncorenlp.VnCoreNLP(annotators=['wseg'], save_dir=str(VNCORE_DIR))
test_text='Tôi đang làm việc tại Đại học Quốc gia Hà Nội.'
test_out=segmenter.word_segment(test_text)
print('Test:', test_out)
assert isinstance(test_out,list) and len(test_out)>0
print('VNCORENLP: PASS')

Downloading: VnCoreNLP-1.2.jar
Downloading: vi-vocab
Downloading: wordsegmenter.rdr
PASS /content/vncorenlp/VnCoreNLP-1.2.jar 26.143 MB
PASS /content/vncorenlp/models/wordsegmenter/vi-vocab 0.502 MB
PASS /content/vncorenlp/models/wordsegmenter/wordsegmenter.rdr 0.123 MB
Test: ['Tôi đang làm_việc tại Đại_học Quốc_gia Hà_Nội .']
VNCORENLP: PASS


In [ ]:
# 6. Segment toàn dataset + cache/resume
SEG_CACHE = CACHE_DIR / 'phobert_segmented.csv'
SEG_META = CACHE_DIR / 'phobert_segmented_metadata.json'

compatible=False
if SEG_CACHE.exists() and SEG_META.exists():
    try:
        smeta=json.loads(SEG_META.read_text(encoding='utf-8'))
        compatible=smeta.get('dataset_sha256')==dataset_sha
    except Exception:
        compatible=False

if not compatible:
    if SEG_CACHE.exists(): SEG_CACHE.unlink()
    if SEG_META.exists(): SEG_META.unlink()
    SEG_META.write_text(json.dumps({'dataset_sha256':dataset_sha,'complete':False},ensure_ascii=False,indent=2),encoding='utf-8')

if SEG_CACHE.exists():
    existing=pd.read_csv(SEG_CACHE).drop_duplicates('row_id',keep='last')
    done=set(existing['row_id'])
else:
    done=set()

buffer=[]
SAVE_EVERY=500

def flush(rows):
    if not rows: return
    chunk=pd.DataFrame(rows)
    chunk.to_csv(SEG_CACHE, mode='a', header=not SEG_CACHE.exists(), index=False)

for row in tqdm(gold.itertuples(index=False), total=len(gold), desc='VnCoreNLP segmentation'):
    if row.row_id in done:
        continue
    segmented=' '.join(segmenter.word_segment(str(row.comment)))
    buffer.append({'row_id':row.row_id,'comment_phobert':segmented})
    done.add(row.row_id)
    if len(buffer)>=SAVE_EVERY:
        flush(buffer); buffer=[]
flush(buffer)

seg=pd.read_csv(SEG_CACHE).drop_duplicates('row_id',keep='last')
assert len(seg)==len(gold) and seg['row_id'].is_unique and set(seg['row_id'])==set(gold['row_id'])
order={rid:i for i,rid in enumerate(gold['row_id'])}
seg['_order']=seg['row_id'].map(order)
seg=seg.sort_values('_order').drop(columns='_order').reset_index(drop=True)
seg.to_csv(SEG_CACHE,index=False)
SEG_META.write_text(json.dumps({'dataset_sha256':dataset_sha,'segmenter':'VnCoreNLP','annotators':['wseg'],'n_rows':len(seg),'complete':True},ensure_ascii=False,indent=2),encoding='utf-8')
TEXT_MAP=dict(zip(seg['row_id'],seg['comment_phobert']))
print('SEGMENTATION COMPLETE:',len(seg))
display(gold[['row_id','comment']].head(3).assign(comment_phobert=lambda x:x['row_id'].map(TEXT_MAP)))

VnCoreNLP segmentation:   0%|          | 0/48670 [00:00<?, ?it/s]

SEGMENTATION COMPLETE: 48670


,row_id,comment,comment_phobert
0,1,"Cá nhân Th_ể thấy fan Sen Vàng bất lịch sự, ng...","Cá_nhân Th _ ể thấy fan Sen_Vàng bất lịch_sự ,..."
1,2,"Fan vô văn hóa thật sự, nghèo còn md","Fan vô văn_hoá thật_sự , nghèo còn md"
2,3,Nếu em không tute thì hãy nhận mức án tuhinh n...,Nếu em không tute thì hãy nhận mức án tuhinh n...


In [ ]:
# 7. Task helpers + metrics

def get_task_frame(split_name,task):
    x=splits[split_name].copy()
    if task=='task2':
        x=x[x['toxicity'].eq('TOXIC')].copy()
    cfg=TASKS[task]
    x['label_id']=x[cfg['label_col']].map(cfg['label2id'])
    x['comment_phobert']=x['row_id'].map(TEXT_MAP)
    assert x['label_id'].notna().all() and x['comment_phobert'].notna().all()
    return x.reset_index(drop=True)

def class_weights(train_df):
    c=train_df['label_id'].value_counts().sort_index(); n=len(train_df); k=len(c)
    return torch.tensor([n/(k*c.loc[i]) for i in range(k)],dtype=torch.float32)

def compute_metrics(y,yp,probs):
    acc=accuracy_score(y,yp)
    mp,mr,mf,_=precision_recall_fscore_support(y,yp,average='macro',zero_division=0)
    _,_,wf,_=precision_recall_fscore_support(y,yp,average='weighted',zero_division=0)
    pr=average_precision_score(y,probs[:,1])
    return {'accuracy':float(acc),'macro_precision':float(mp),'macro_recall':float(mr),'macro_f1':float(mf),'weighted_f1':float(wf),'pr_auc_positive_class':float(pr)}

def per_class_df(y,yp,id2label):
    p,r,f,s=precision_recall_fscore_support(y,yp,labels=[0,1],average=None,zero_division=0)
    return pd.DataFrame([{'label':id2label[i],'precision':float(p[i]),'recall':float(r[i]),'f1':float(f[i]),'support':int(s[i])} for i in [0,1]])

def save_json(obj,path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')

for task in ['task1','task2']:
    print(); print(task.upper())
    for s in ['train','dev','test']:
        x=get_task_frame(s,task)
        print(s,len(x),x['label_id'].value_counts().sort_index().to_dict())


TASK1
train 38936 {0: 34248, 1: 4688}
dev 4867 {0: 4281, 1: 586}
test 4867 {0: 4281, 1: 586}

TASK2
train 4688 {0: 3999, 1: 689}
dev 586 {0: 500, 1: 86}
test 586 {0: 500, 1: 86}


In [ ]:
# 8. Tokenizer + forward/backward smoke test
# PhoBERT model card yêu cầu word-segmented input; dùng slow PhoBertTokenizer chính thức.
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=False)
sample=get_task_frame('train','task1').head(2)
enc=tokenizer(sample['comment_phobert'].tolist(),truncation=True,padding=True,max_length=64,return_tensors='pt')
model=AutoModelForSequenceClassification.from_pretrained(MODEL_ID,num_labels=2).to(DEVICE)
enc_gpu={k:v.to(DEVICE) for k,v in enc.items()}
out=model(**enc_gpu)
assert out.logits.shape==(2,2)
out.logits.sum().backward()
print('Tokenizer:',type(tokenizer).__name__)
print('SMOKE TEST: PASS',tuple(out.logits.shape))
del model,out,enc,enc_gpu; gc.collect(); torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizer: PhobertTokenizer
SMOKE TEST: PASS (2, 2)


In [ ]:
# 9. Token-length analysis chỉ trên TRAIN, riêng Task 1 và Task 2
TOKEN_FILE=RESULT_DIR/'token_length_analysis.csv'

def token_lengths(texts,batch=256):
    lens=[]
    for start in tqdm(range(0,len(texts),batch),desc='Token lengths',leave=False):
        e=tokenizer(texts[start:start+batch],add_special_tokens=True,truncation=False,padding=False)
        lens.extend(len(ids) for ids in e['input_ids'])
    return np.asarray(lens)

def choose_max(lens):
    for c in [64,128,192,256]:
        if np.mean(lens<=c)>=0.95: return c
    return 256

rows=[]
for task in ['task1','task2']:
    tr=get_task_frame('train',task)
    lens=token_lengths(tr['comment_phobert'].astype(str).tolist())
    rows.append({'model':'phobert','task':task,'n_samples':len(lens),'p50':float(np.quantile(lens,.5)),'p90':float(np.quantile(lens,.9)),'p95':float(np.quantile(lens,.95)),'p99':float(np.quantile(lens,.99)),'max':int(lens.max()),'coverage_64':float(np.mean(lens<=64)),'coverage_128':float(np.mean(lens<=128)),'coverage_192':float(np.mean(lens<=192)),'coverage_256':float(np.mean(lens<=256)),'selected_max_length':int(choose_max(lens))})

token_stats=pd.DataFrame(rows)
if TOKEN_FILE.exists():
    old=pd.read_csv(TOKEN_FILE)
    old=old[old['model'].astype(str).str.lower()!='phobert']
    all_stats=pd.concat([old,token_stats],ignore_index=True)
else:
    all_stats=token_stats.copy()
all_stats.to_csv(TOKEN_FILE,index=False)
display(token_stats)

Token lengths:   0%|          | 0/153 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Token lengths:   0%|          | 0/19 [00:00<?, ?it/s]

,model,task,n_samples,p50,p90,p95,p99,max,coverage_64,coverage_128,coverage_192,coverage_256,selected_max_length
0,phobert,task1,38936,14.0,48.0,71.00,129.00,1021,0.940621,0.989624,0.999050,0.999743,128
1,phobert,task2,4688,21.0,68.0,96.65,155.13,320,0.887799,0.976962,0.997227,0.999360,128


In [ ]:
# 10. Pre-tokenized Dataset + evaluation
class EncodedDataset(Dataset):
    def __init__(self,encodings,labels):
        self.encodings=encodings
        self.labels=torch.tensor(labels,dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        item={k:v[i] for k,v in self.encodings.items()}
        item['labels']=self.labels[i]
        return item

def encode_frame(frame,max_length):
    enc=tokenizer(frame['comment_phobert'].astype(str).tolist(),truncation=True,padding='max_length',max_length=max_length,return_tensors='pt')
    return EncodedDataset(enc,frame['label_id'].tolist())

@torch.no_grad()
def evaluate(model,loader,loss_fn):
    model.eval(); losses=[]; logits_all=[]; y_all=[]
    for batch in loader:
        labels=batch.pop('labels').to(DEVICE,non_blocking=True)
        inputs={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}
        with torch.autocast('cuda',dtype=torch.float16):
            logits=model(**inputs).logits
            loss=loss_fn(logits,labels)
        losses.append(loss.item()); logits_all.append(logits.cpu()); y_all.append(labels.cpu())
    logits=torch.cat(logits_all); y=torch.cat(y_all).numpy()
    probs=torch.softmax(logits.float(),dim=1).numpy(); yp=probs.argmax(1)
    m=compute_metrics(y,yp,probs); m['loss']=float(np.mean(losses))
    return m,y,yp,probs

In [ ]:
# 11. Full fine-tuning: 10 epochs max + patience 3 + best DEV Macro-F1

def train_task(task,max_length,batch_size,grad_accum):
    cfg=TASKS[task]
    out_dir=ARTIFACT_DIR/MODEL_KEY/task
    best_dir=out_dir/'best_model'; tok_dir=out_dir/'tokenizer'
    pred_path=PRED_DIR/f'{MODEL_KEY}_{task}.csv'; cm_path=CM_DIR/f'{MODEL_KEY}_{task}_cm.csv'
    completed=[out_dir/'test_metrics.json',out_dir/'dev_metrics.json',out_dir/'training_history.csv',out_dir/'experiment_config.json',pred_path,cm_path]
    if all(p.exists() for p in completed):
        print('SKIP COMPLETED:',task); return
    if out_dir.exists(): shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True,exist_ok=True)

    frames={s:get_task_frame(s,task) for s in ['train','dev']}
    datasets={s:encode_frame(frames[s],max_length) for s in frames}
    loaders={s:DataLoader(datasets[s],batch_size=batch_size,shuffle=(s=='train'),num_workers=0,pin_memory=True) for s in datasets}

    w=class_weights(frames['train']).to(DEVICE)
    seed_all()
    model=AutoModelForSequenceClassification.from_pretrained(MODEL_ID,num_labels=2,id2label=cfg['id2label'],label2id=cfg['label2id']).to(DEVICE)
    loss_fn=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    updates_epoch=math.ceil(len(loaders['train'])/grad_accum)
    total_updates=updates_epoch*MAX_EPOCHS
    scheduler=get_cosine_schedule_with_warmup(opt,num_warmup_steps=int(total_updates*WARMUP_RATIO),num_training_steps=total_updates)
    try: scaler=torch.amp.GradScaler('cuda',enabled=True)
    except Exception: scaler=torch.cuda.amp.GradScaler(enabled=True)

    best=-1.; best_epoch=None; wait=0; hist=[]; started=time.time(); opt.zero_grad(set_to_none=True)
    for epoch in range(1,MAX_EPOCHS+1):
        model.train(); train_losses=[]
        bar=tqdm(loaders['train'],desc=f'PhoBERT {task} epoch {epoch}/{MAX_EPOCHS}')
        for step,batch in enumerate(bar,1):
            labels=batch.pop('labels').to(DEVICE,non_blocking=True)
            inputs={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}
            with torch.autocast('cuda',dtype=torch.float16):
                logits=model(**inputs).logits
                raw_loss=loss_fn(logits,labels)
                loss=raw_loss/grad_accum
            scaler.scale(loss).backward(); train_losses.append(raw_loss.item())
            if step%grad_accum==0 or step==len(loaders['train']):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                scaler.step(opt); scaler.update(); scheduler.step(); opt.zero_grad(set_to_none=True)
            if step%100==0: bar.set_postfix(loss=f'{np.mean(train_losses[-100:]):.4f}')

        dm,_,_,_=evaluate(model,loaders['dev'],loss_fn)
        row={'epoch':epoch,'train_loss':float(np.mean(train_losses)),'dev_loss':dm['loss'],'dev_accuracy':dm['accuracy'],'dev_macro_precision':dm['macro_precision'],'dev_macro_recall':dm['macro_recall'],'dev_macro_f1':dm['macro_f1'],'dev_weighted_f1':dm['weighted_f1'],'dev_pr_auc_positive':dm['pr_auc_positive_class']}
        hist.append(row); print(json.dumps(row,ensure_ascii=False,indent=2))

        if dm['macro_f1']>best+1e-6:
            best=dm['macro_f1']; best_epoch=epoch; wait=0
            if best_dir.exists(): shutil.rmtree(best_dir)
            if tok_dir.exists(): shutil.rmtree(tok_dir)
            model.save_pretrained(best_dir,safe_serialization=True); tokenizer.save_pretrained(tok_dir)
            print('Saved best epoch',best_epoch,'dev_macro_f1=',round(best,6))
        else:
            wait+=1; print('No improvement',f'{wait}/{PATIENCE}')
            if wait>=PATIENCE:
                print('EARLY STOPPING'); break

    seconds=time.time()-started
    pd.DataFrame(hist).to_csv(out_dir/'training_history.csv',index=False)
    del model; gc.collect(); torch.cuda.empty_cache()

    model=AutoModelForSequenceClassification.from_pretrained(best_dir).to(DEVICE)
    best_dev,_,_,_=evaluate(model,loaders['dev'],loss_fn)

    # Chỉ lúc này mới tạo Test dataset/loader: checkpoint đã khóa bằng Dev.
    test_frame=get_task_frame('test',task)
    test_dataset=encode_frame(test_frame,max_length)
    test_loader=DataLoader(test_dataset,batch_size=batch_size,shuffle=False,num_workers=0,pin_memory=True)
    test_m,y,yp,probs=evaluate(model,test_loader,loss_fn)

    pred=test_frame[['row_id','comment']].copy(); id2=cfg['id2label']
    pred['gold_label']=[id2[int(i)] for i in y]; pred['predicted_label']=[id2[int(i)] for i in yp]
    pred['prob_class_0']=probs[:,0]; pred['prob_class_1']=probs[:,1]; pred['correct']=pred['gold_label'].eq(pred['predicted_label'])
    pred.to_csv(pred_path,index=False)
    pc=per_class_df(y,yp,id2); pc.to_csv(out_dir/'classification_report.csv',index=False)
    cm=confusion_matrix(y,yp,labels=[0,1]); cm_df=pd.DataFrame(cm,index=[id2[0],id2[1]],columns=[id2[0],id2[1]]); cm_df.to_csv(cm_path)

    env={'python':sys.version,'torch':torch.__version__,'transformers':transformers.__version__,'sklearn':sklearn.__version__,'pandas':pd.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)}
    cfg_out={'model_key':MODEL_KEY,'model_id':MODEL_ID,'task':task,'seed':SEED,'max_epochs':MAX_EPOCHS,'early_stopping_patience':PATIENCE,'best_checkpoint_metric':'dev_macro_f1','best_epoch':best_epoch,'max_length':int(max_length),'physical_batch_size':int(batch_size),'gradient_accumulation_steps':int(grad_accum),'effective_batch_size':int(batch_size*grad_accum),'learning_rate':LR,'weight_decay':WEIGHT_DECAY,'warmup_ratio':WARMUP_RATIO,'scheduler':'cosine_with_warmup','gradient_clip':GRAD_CLIP,'mixed_precision':'fp16','loss':'weighted_cross_entropy','class_weights':w.cpu().tolist(),'dataset_sha256':dataset_sha,'word_segmentation':'VnCoreNLP wseg','training_time_seconds':float(seconds),'environment':env}
    save_json(cfg_out,out_dir/'experiment_config.json'); save_json(best_dev,out_dir/'dev_metrics.json'); save_json(test_m,out_dir/'test_metrics.json')
    print(); print('COMPLETED',task,'BEST EPOCH',best_epoch); print('DEV',best_dev); print('TEST',test_m); display(pc); display(cm_df)
    del model,loaders,datasets,test_loader,test_dataset; gc.collect(); torch.cuda.empty_cache()

In [12]:
# 12. OOM-safe run: effective batch luôn xấp xỉ 32

def run_task(task,max_length):
    last=None
    for batch,accum in [(16,2),(8,4),(4,8)]:
        try:
            print(); print('Attempt',task,'batch=',batch,'accum=',accum,'effective=',batch*accum)
            train_task(task,max_length,batch,accum)
            return
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower(): raise
            last=str(e); traceback.clear_frames(e.__traceback__); print('CUDA OOM → retry batch nhỏ hơn'); gc.collect(); torch.cuda.empty_cache()
    raise RuntimeError(f'Không train được {task} do OOM. Last error: {last}')

for task in ['task1','task2']:
    max_len=int(token_stats.loc[token_stats['task'].eq(task),'selected_max_length'].iloc[0])
    print(); print('#'*90); print('PHOBERT',task.upper(),'max_length=',max_len); print('#'*90)
    run_task(task,max_len)


##########################################################################################
PHOBERT TASK1 max_length= 128
##########################################################################################

Attempt task1 batch= 16 accum= 2 effective= 32


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PhoBERT task1 epoch 1/10:   0%|          | 0/2434 [00:00<?, ?it/s]

{
  "epoch": 1,
  "train_loss": 0.49328694935817646,
  "dev_loss": 0.37549951764159517,
  "dev_accuracy": 0.8409697965892747,
  "dev_macro_precision": 0.695006140997298,
  "dev_macro_recall": 0.8322736067694942,
  "dev_macro_f1": 0.7286866359447004,
  "dev_weighted_f1": 0.8611958274431679,
  "dev_pr_auc_positive": 0.678035036811597
}
Saved best epoch 1 dev_macro_f1= 0.728687


PhoBERT task1 epoch 2/10:   0%|          | 0/2434 [00:00<?, ?it/s]

{
  "epoch": 2,
  "train_loss": 0.3601130110976166,
  "dev_loss": 0.3463594533258774,
  "dev_accuracy": 0.8639819190466407,
  "dev_macro_precision": 0.7192537912096388,
  "dev_macro_recall": 0.8512462400335477,
  "dev_macro_f1": 0.7572762419618266,
  "dev_weighted_f1": 0.8794570528785876,
  "dev_pr_auc_positive": 0.7083886510113443
}
Saved best epoch 2 dev_macro_f1= 0.757276


... (26 outputs left out)

PhoBERT task2 epoch 6/10:   0%|          | 0/293 [00:00<?, ?it/s]

{
  "epoch": 6,
  "train_loss": 0.16268302727198233,
  "dev_loss": 1.1514497924898122,
  "dev_accuracy": 0.8174061433447098,
  "dev_macro_precision": 0.6386973526552574,
  "dev_macro_recall": 0.6426744186046511,
  "dev_macro_f1": 0.6406247313368678,
  "dev_weighted_f1": 0.81869651861484,
  "dev_pr_auc_positive": 0.47231577934807534
}
No improvement 3/3
EARLY STOPPING

COMPLETED task2 BEST EPOCH 3
DEV {'accuracy': 0.810580204778157, 'macro_precision': 0.6693356327502669, 'macro_recall': 0.7397674418604652, 'macro_f1': 0.6905091568294388, 'weighted_f1': 0.8266993317630533, 'pr_auc_positive_class': 0.42881699197829365, 'loss': 0.5896966276136605}
TEST {'accuracy': 0.8003412969283277, 'macro_precision': 0.651279255096049, 'macro_recall': 0.7096976744186047, 'macro_f1': 0.669158202041161, 'weighted_f1': 0.8163392353292016, 'pr_auc_positive_class': 0.5015870013247824, 'loss': 0.5801606846822275}


,label,precision,recall,f1,support
0,EXPLICIT,0.920879,0.838000,0.877487,500
1,IMPLICIT,0.381679,0.581395,0.460829,86


,EXPLICIT,IMPLICIT
EXPLICIT,419,81
IMPLICIT,36,50


In [13]:
# 13. Final QA + bảng kết quả PhoBERT
qa=[]; summary=[]
for task in ['task1','task2']:
    out=ARTIFACT_DIR/'phobert'/task
    checks={'task':task,'best_model':(out/'best_model').exists(),'tokenizer':(out/'tokenizer').exists(),'history':(out/'training_history.csv').exists(),'dev_metrics':(out/'dev_metrics.json').exists(),'test_metrics':(out/'test_metrics.json').exists(),'config':(out/'experiment_config.json').exists(),'per_class':(out/'classification_report.csv').exists(),'prediction':(PRED_DIR/f'phobert_{task}.csv').exists(),'confusion_matrix':(CM_DIR/f'phobert_{task}_cm.csv').exists()}
    qa.append(checks)
    pred=pd.read_csv(PRED_DIR/f'phobert_{task}.csv'); expected_test=get_task_frame('test',task)
    assert pred['row_id'].is_unique and set(pred['row_id'])==set(expected_test['row_id']) and len(pred)==len(expected_test)
    dm=json.loads((out/'dev_metrics.json').read_text(encoding='utf-8')); tm=json.loads((out/'test_metrics.json').read_text(encoding='utf-8')); cfg=json.loads((out/'experiment_config.json').read_text(encoding='utf-8'))
    summary.append({'Task':task,'Best_Epoch':cfg['best_epoch'],'Max_Length':cfg['max_length'],'Batch':cfg['physical_batch_size'],'Grad_Accum':cfg['gradient_accumulation_steps'],'Dev_Macro_F1':dm['macro_f1'],'Test_Accuracy':tm['accuracy'],'Test_Macro_P':tm['macro_precision'],'Test_Macro_R':tm['macro_recall'],'Test_Macro_F1':tm['macro_f1'],'Test_Weighted_F1':tm['weighted_f1'],'Test_PR_AUC_Positive':tm['pr_auc_positive_class']})

qa_df=pd.DataFrame(qa); display(qa_df)
assert qa_df.drop(columns=['task']).all(axis=None), 'PhoBERT QA chưa PASS.'
print('PHOBERT FINAL QA: PASS')
display(pd.DataFrame(summary))
print('Artifacts:',ARTIFACT_DIR/'phobert')
print('Predictions:',PRED_DIR)

,task,best_model,tokenizer,history,dev_metrics,test_metrics,config,per_class,prediction,confusion_matrix
0,task1,True,True,True,True,True,True,True,True,True
1,task2,True,True,True,True,True,True,True,True,True


PHOBERT FINAL QA: PASS


,Task,Best_Epoch,Max_Length,Batch,Grad_Accum,Dev_Macro_F1,Test_Accuracy,Test_Macro_P,Test_Macro_R,Test_Macro_F1,Test_Weighted_F1,Test_PR_AUC_Positive
0,task1,5,128,16,2,0.800657,0.917608,0.812940,0.778627,0.794342,0.915220,0.694329
1,task2,3,128,16,2,0.690509,0.800341,0.651279,0.709698,0.669158,0.816339,0.501587


Artifacts: /content/drive/MyDrive/DS108_Toxic_4Models/artifacts/phobert
Predictions: /content/drive/MyDrive/DS108_Toxic_4Models/results/predictions


## Kết quả cần thấy ở cuối notebook

`PHOBERT FINAL QA: PASS`

Artifacts được lưu tại:

```text
MyDrive/DS108_Toxic_4Models/
├── artifacts/phobert/task1/
├── artifacts/phobert/task2/
├── results/predictions/phobert_task1.csv
├── results/predictions/phobert_task2.csv
├── results/confusion_matrices/phobert_task1_cm.csv
├── results/confusion_matrices/phobert_task2_cm.csv
├── results/token_length_analysis.csv
└── cache/phobert_segmented.csv
```

Sau khi PASS, chuyển sang `03_ViDeBERTa.ipynb`.